Using the output of pathway importer, fill the PATHWAY and the PATHWAY_TO_PROTEIN tables.

TODO: This cannot have worked for mouse because you have not imported mouse proteins yet... 

In [1]:
import pandas as pd
import json
from tqdm import tqdm
from pathlib import Path
import sqlite3

First, we collect a list of pathways as well as pathways to proteins

In [2]:
organism_taxcodes = [9606, 10090]

In [3]:
json_directory = Path('../resources/wikipathways_jsons/')

In [4]:
GENE_NODETYPE = 'gene_protein'

In [5]:
database = 'wikipathways'

In [6]:
pathway_tuples = []
pathway_to_protein_tuples = []
pathway_index = 1

In [7]:
for taxcode in tqdm(organism_taxcodes, 'Organisms'):
    json_directory_organism = json_directory.joinpath(str(taxcode))
    for jsonfile in tqdm(list(json_directory_organism.glob('*.json')), 'Pathways'):
        with open(jsonfile, 'r') as infile:
            pathway_skeleton = json.load(infile)
        # This splits the 'path:' prefix away from KEGG pathways
        pathway_name = pathway_skeleton['pathway']['name'].split(':')[-1]
        pathway_title = pathway_skeleton['pathway']['title']
        pathway_n_genes = len(
            {",".join(sorted(node['geneNames']))
             for node in pathway_skeleton['nodes'] if node['type'] == GENE_NODETYPE})
        # Pathways with 0 genes are not interesting for this tool, since no peptide will ever be mapped to them
        if pathway_n_genes == 0:
            continue
        pathway_tuples.append((pathway_name, pathway_title, taxcode, database, pathway_n_genes, json.dumps(pathway_skeleton)))
        for node in pathway_skeleton['nodes']:
            if node['type'] == GENE_NODETYPE:
                for gene in node.get('geneNames') or []:
                    if gene:
                        pathway_to_protein_tuples.append(
                            (pathway_index, node.get('id'), gene, 'GENE_SYMBOL', taxcode))
                for uniprotAccs in node.get('uniprotAccs') or []:
                    if uniprotAccs:
                        pathway_to_protein_tuples.append(
                            (pathway_index, node.get('id'), uniprotAccs, 'UNIPROT', taxcode))
        pathway_index += 1

Organisms: 100%|███████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  3.20it/s]


In [8]:
pathway_df = pd.DataFrame(pathway_tuples,
                          columns=['PATHWAY_NAME', 'TITLE', 'TAXCODE', 'DATABASE', 'N_GENES', 'PATHWAY_JSON'])
pathway_df.index.name = 'PATHWAY_ID'
# SQL Tables are conventionally 1-indexed
pathway_df.index += 1
pathway_df

,PATHWAY_NAME,TITLE,TAXCODE,DATABASE,N_GENES,PATHWAY_JSON
PATHWAY_ID,,,,,,
1,WP500,Glycogen synthesis and degradation,9606,wikipathways,45,"{""pathway"": {""name"": ""WP500"", ""org"": ""Homo sap..."
2,WP23,B cell receptor signaling pathway,9606,wikipathways,103,"{""pathway"": {""name"": ""WP23"", ""org"": ""Homo sapi..."
3,WP5238,Cholestasis,9606,wikipathways,20,"{""pathway"": {""name"": ""WP5238"", ""org"": ""Homo sa..."
4,WP550,Biogenic amine synthesis,9606,wikipathways,15,"{""pathway"": {""name"": ""WP550"", ""org"": ""Homo sap..."
5,WP3407,FTO obesity variant mechanism,9606,wikipathways,8,"{""pathway"": {""name"": ""WP3407"", ""org"": ""Homo sa..."
...,...,...,...,...,...,...
1084,WP456,"GPCRs, class B secretin-like",10090,wikipathways,23,"{""pathway"": {""name"": ""WP456"", ""org"": ""Mus musc..."
1085,WP373,IL-3 signaling pathway,10090,wikipathways,102,"{""pathway"": {""name"": ""WP373"", ""org"": ""Mus musc..."
1086,WP570,Monoamine GPCRs,10090,wikipathways,33,"{""pathway"": {""name"": ""WP570"", ""org"": ""Mus musc..."


In [9]:
pathway_to_protein_df = pd.DataFrame(pathway_to_protein_tuples,
                                     columns=['PATHWAY_ID', 'NODE_ID', 'GENE_IDENTIFIER', 'IDENTIFIER_TYPE',
                                              'TAXCODE'])

# Concatenate the NODE_IDs of identical genes into lists so that each gene identifier is unique in each pathway
pathway_to_protein_df = pathway_to_protein_df.groupby(
    ['PATHWAY_ID', 'GENE_IDENTIFIER', 'IDENTIFIER_TYPE', 'TAXCODE']).agg(
    lambda node_ids: ",".join(list(node_ids))).reset_index()
pathway_to_protein_df.rename({'NODE_ID': 'NODE_IDS'}, axis=1, inplace=True)
pathway_to_protein_df

,PATHWAY_ID,GENE_IDENTIFIER,IDENTIFIER_TYPE,TAXCODE,NODE_IDS
0,1,A0A0S2A4E4,UNIPROT,9606,df1
1,1,A0A140VJS0,UNIPROT,9606,a2e
2,1,A0A140VJT0,UNIPROT,9606,e1e
3,1,A0A140VKE1,UNIPROT,9606,"b6f,ced"
4,1,A8K7B7,UNIPROT,9606,c15
...,...,...,...,...,...
202748,1088,Rxrb,GENE_SYMBOL,10090,a09
202749,1088,Rxrg,GENE_SYMBOL,10090,ca3
202750,1088,Thra,GENE_SYMBOL,10090,c8b
202751,1088,Thrb,GENE_SYMBOL,10090,"db9,c5e"


Insert them into the database

In [11]:
conn = sqlite3.connect('../sqlite_backend.db')

In [12]:
pathway_df.reset_index().to_sql('PATHWAY', conn, if_exists='append', index=False)

1088

In [13]:
protein_df = pd.read_sql('SELECT PROTEIN_ID, GENE_NAME, UNIPROT_ACC FROM PROTEIN', conn) 

In [19]:
pathway_to_protein_df

,PATHWAY_ID,GENE_IDENTIFIER,IDENTIFIER_TYPE,TAXCODE,NODE_IDS
0,1,A0A0S2A4E4,UNIPROT,9606,df1
1,1,A0A140VJS0,UNIPROT,9606,a2e
2,1,A0A140VJT0,UNIPROT,9606,e1e
3,1,A0A140VKE1,UNIPROT,9606,"b6f,ced"
4,1,A8K7B7,UNIPROT,9606,c15
...,...,...,...,...,...
202748,1088,Rxrb,GENE_SYMBOL,10090,a09
202749,1088,Rxrg,GENE_SYMBOL,10090,ca3
202750,1088,Thra,GENE_SYMBOL,10090,c8b
202751,1088,Thrb,GENE_SYMBOL,10090,"db9,c5e"


In [29]:
pw2pr_genesymbol = pathway_to_protein_df[pathway_to_protein_df['IDENTIFIER_TYPE'] == 'GENE_SYMBOL'].merge(
    protein_df,
    left_on = 'GENE_IDENTIFIER',
    right_on = 'GENE_NAME',
    how='inner')[['PATHWAY_ID', 'NODE_IDS', 'PROTEIN_ID']]
pw2pr_genesymbol

,PATHWAY_ID,NODE_IDS,PROTEIN_ID
0,1,df1,15524
1,1,df1,64632
2,1,df1,64633
3,1,df1,65982
4,1,af6,12224
...,...,...,...
239326,1087,e51d5,1024
239327,1087,e51d5,29715
239328,1087,e51d5,29718
239329,1087,e51d5,57442


In [30]:
pw2pr_uniprot = pathway_to_protein_df[pathway_to_protein_df['IDENTIFIER_TYPE'] == 'UNIPROT'].merge(
    protein_df,
    left_on = 'GENE_IDENTIFIER',
    right_on = 'UNIPROT_ACC',
    how='inner')[['PATHWAY_ID', 'NODE_IDS', 'PROTEIN_ID']]
pw2pr_uniprot

,PATHWAY_ID,NODE_IDS,PROTEIN_ID
0,1,f57,41593
1,1,d29,3212
2,1,"bbcb8,fd9,c7cc1",4820
3,1,"af6,c2a,df8",12224
4,1,"af6,c2a,df8",3721
...,...,...,...
121743,1030,c61c3,8857
121744,1043,bce78,7415
121745,1043,d1244,13920
121746,1046,b7b,18828


In [31]:
pw2pr_concatenated = pd.concat([pw2pr_uniprot, pw2pr_genesymbol]).drop_duplicates()
pw2pr_concatenated

,PATHWAY_ID,NODE_IDS,PROTEIN_ID
0,1,f57,41593
1,1,d29,3212
2,1,"bbcb8,fd9,c7cc1",4820
3,1,"af6,c2a,df8",12224
4,1,"af6,c2a,df8",3721
...,...,...,...
239326,1087,e51d5,1024
239327,1087,e51d5,29715
239328,1087,e51d5,29718
239329,1087,e51d5,57442


In [34]:
pw2pr_concatenated.to_sql('PATHWAY_TO_PROTEIN', conn, if_exists='append', index=False)

254997